# FLUX-LM Universal: Multi-Domain Byte Language Model Training

**Training a vocabulary-free language model (500M-1B) on multiple domains simultaneously.**

> **Note:** This notebook is designed for **Google Colab** with GPU runtime. Checkpoints are saved to Google Drive.

## Dynamic Model Selection (VRAM-based)
| VRAM | Model | Batch | Seq Len |
|------|-------|-------|---------|
| **40GB+** (A100, H100) | **1B** | 4 | 1024 |
| **22-40GB** (L4, RTX 4090) | **500M** | 2 | 512 |
| **<22GB** | ❌ Error | - | - |

## Architecture Comparison
| Component | 500M | 1B |
|-----------|------|----| 
| CSE | ~30M | ~50M |
| **CWC** (order-aware) | ~15M | ~25M |
| WavePredictor | ~400M | ~800M |
| Decoder | ~30M | ~50M |
| **Total** | **~480M** | **~930M** |

## Training Features
- ✅ **CWC Order Training** — Auxiliary loss for sequence order discrimination
- ✅ **Validation Generation Demos** — Shows model output at each checkpoint
- ✅ **Google Drive Saving** — All checkpoints persisted to Drive
- ✅ Aggressive disk/memory management
- ✅ Bounded memory (5K train losses max)

## Training Data (~2B tokens)
| Category | Samples | Est. Tokens |
|----------|---------|-------------|
| Assistant | ~227K | ~450M |
| Reasoning | ~14K | ~50M |
| Code | ~501K | ~750M |
| Multilingual | ~500K | ~500M |
| Docs | ~100K | ~200M |
| **Total** | **~1.34M** | **~2B** |

## Estimated Training Time
| GPU | Model | Est. Time |
|-----|-------|-----------|
| A100 (40GB) | 1B | ~12 hours |
| A100 (80GB) | 1B | ~6 hours |
| H100 (80GB) | 1B | ~4 hours |
| L4 (22.5GB) | 500M | ~10 hours |

In [ ]:
# Cell 1: Setup and Imports
import os
import sys
import gc
import math
import random
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from tqdm.auto import tqdm

# Check environment
print("=" * 60)
print("FLUX-LM Universal: Multi-Domain Training (Colab)")
print("=" * 60)

# ==== COLAB-ONLY ENVIRONMENT ====
# This notebook is designed for Google Colab with GPU runtime
IN_COLAB = 'google.colab' in sys.modules

if not IN_COLAB:
    # Local development
    ROOT = Path('/Users/admin/Desktop/flux')
    SAVE_TO_DRIVE = False
    DRIVE_CKPT_DIR = None
    print("Environment: Local (no Drive saving)")
else:
    # Google Colab
    ROOT = Path('/content/FLUX')
    print("Environment: Google Colab")
    
    # ==== GOOGLE DRIVE SETUP ====
    # Mount Google Drive for persistent checkpoint storage
    SAVE_TO_DRIVE = True
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create checkpoint directory on Drive
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/FLUX_checkpoints')
    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✓ Google Drive mounted")
    print(f"✓ Checkpoints will be saved to: {DRIVE_CKPT_DIR}")

print(f"Root: {ROOT}")

In [ ]:
# Cell 2: Clone/Update Repository
if IN_COLAB:
    if not ROOT.exists():
        print("Cloning FLUX repository...")
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    else:
        print("Updating FLUX repository...")
        %cd {ROOT}
        !git pull
    
    # Install dependencies
    !pip install -q datasets tqdm rich matplotlib

%cd {ROOT}
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'phases' / 'phase_lm'))

print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 3: Hardware Detection and Configuration
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

print(f"\nDevice: {device}")

if device != 'cuda':
    raise RuntimeError("This notebook requires CUDA GPU with 22GB+ VRAM (use Colab with GPU runtime)")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")

# ==== DYNAMIC MODEL SELECTION: 500M or 1B ====
# Auto-select based on VRAM (minimum 500M, no smaller models)

if gpu_mem >= 40:
    # A100 40GB/80GB, H100: Use 1B
    MODEL_SCALE = '1B'
    BATCH_SIZE = 4
    MAX_SEQ_LEN = 1024
    GRAD_ACCUM = 16
    print(f"\n✓ VRAM sufficient for 1B model")
elif gpu_mem >= 22:
    # L4 (22.5GB), RTX 4090, A10: Use 500M with conservative settings
    MODEL_SCALE = '500M'
    BATCH_SIZE = 2       # Reduced from 4 to avoid OOM
    MAX_SEQ_LEN = 512
    GRAD_ACCUM = 32      # Increased to maintain effective batch of 64
    print(f"\n✓ VRAM sufficient for 500M model (conservative settings)")
else:
    raise RuntimeError(
        f"Insufficient VRAM: {gpu_mem:.1f}GB\n"
        f"Minimum 22GB required for 500M model.\n"
        f"Use Colab with L4/A100 GPU runtime."
    )

print(f"\n{'='*50}")
print(f"Selected model: FLUX-LM Universal {MODEL_SCALE}")
print(f"{'='*50}")
print(f"Training config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max seq len: {MAX_SEQ_LEN}")
print(f"  Grad accum: {GRAD_ACCUM}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

# ==== GPU OPTIMIZATIONS ====
torch.backends.cudnn.benchmark = True
print("  ✓ cuDNN benchmark enabled")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("  ✓ TF32 precision enabled")

if hasattr(torch.nn.functional, 'scaled_dot_product_attention'):
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    print("  ✓ Flash/Memory-efficient attention enabled")

DATALOADER_WORKERS = 4
DATALOADER_PREFETCH = 4
PERSISTENT_WORKERS = True
PIN_MEMORY = True
print(f"  DataLoader: workers={DATALOADER_WORKERS}, prefetch={DATALOADER_PREFETCH}")

In [ ]:
# Cell 4: Import FLUX-LM Universal Components

from flux_lm_universal import (
    FluxLMUniversal, 
    GenerationConfigUniversal,
    FLUX_LM_UNIVERSAL_CONFIG,
    FLUX_LM_500M_CONFIG,
    FLUX_LM_1B_CONFIG,
    FLUX_LM_3B_CONFIG,
    format_params,
    DomainEmbedding,
)

from multi_domain_data import (
    DATASET_CONFIGS,
    SPECIAL_TOKENS,
    VOCAB_SIZE,
    load_all_datasets,
    create_dataloaders,
    encode_special,
    decode_special,
    MultiDomainDataset,
    DomainDataset,
)

print("✓ FLUX-LM Universal modules imported!")
print(f"\nAvailable datasets ({len(DATASET_CONFIGS)}):")
for name, cfg in DATASET_CONFIGS.items():
    print(f"  - {name}: {cfg.domain} (max {cfg.max_samples:,} samples, weight {cfg.weight:.2f})")

In [ ]:
# Cell 5: Create Model

print(f"Creating FLUX-LM Universal model (scale: {MODEL_SCALE})...")

model = FluxLMUniversal(scale=MODEL_SCALE)
model = model.to(device)

# Parameter counts
params = model.count_parameters()
print("\n" + "=" * 50)
print(f"FLUX-LM Universal {MODEL_SCALE} Parameter Breakdown:")
print("=" * 50)
for name, count in params.items():
    print(f"  {name:15s}: {format_params(count):>10s}")
print("=" * 50)

total = params['total']
print(f"\n✓ Model created: {format_params(total)} parameters")
print(f"  Vocabulary size: {model.vocab_size} (256 bytes + {model.vocab_size - 256} special tokens)")

## Multi-Domain Dataset Loading

Loading datasets with **100% sampling** for proper model training (~2B tokens):

| Domain | Dataset | Samples | Est. Tokens |
|--------|---------|---------|-------------|
| **Reasoning** | Opus, GSM8K, ARC | 14K | ~50M |
| **Assistant** | OpenAssistant, Dolly, Alpaca | 227K | ~450M |
| **Code** | CodeSearchNet, HumanEval, MBPP | 501K | ~750M |
| **Multilingual** | OPUS-100 (5 langs × 100K) | 500K | ~500M |
| **Docs** | WikiText-103 | 100K | ~200M |
| **TOTAL** | | **~1.34M** | **~2B** |

**Chinchilla Scaling:**
- 500M × 20 = 10B tokens optimal → ~2B = 20% (good)
- 1B × 20 = 20B tokens optimal → ~2B = 10% (acceptable)

In [ ]:
# Cell 6: Select Datasets to Load (100% for 500M-1B)

# ==== DATASET SELECTION ====
# ALL datasets loaded at 100% capacity for 500M-1B model training
# Total: ~1.34M samples, ~2B tokens

DATASETS_TO_LOAD = [
    # Priority 1: Reasoning (14K samples)
    'opus_reasoning',      # 3K deep reasoning chains
    'gsm8k',              # 8.5K math problems
    'arc_challenge',      # 2.5K science reasoning
    
    # Priority 1: Assistant (227K samples)
    'openassistant',      # 160K conversations (100%)
    'dolly',              # 15K instruction-response
    'alpaca',             # 52K (100%)
    
    # Priority 1: Code (501K samples)
    'humaneval',          # 164 code problems
    'mbpp',               # 974 code problems
    'code_search_net',    # 500K Python functions (100%)
    
    # Priority 2: Multilingual (500K samples)
    'opus100_en_fr',      # 100K EN-FR translations
    'opus100_en_de',      # 100K EN-DE translations
    'opus100_en_zh',      # 100K EN-ZH translations
    'opus100_en_es',      # 100K EN-ES translations
    'opus100_en_ja',      # 100K EN-JA translations
    
    # Priority 2: Docs (100K samples)
    'wikitext',           # 100K wiki articles
]

print(f"Selected {len(DATASETS_TO_LOAD)} datasets for {MODEL_SCALE} model training:")
print(f"\n{'Dataset':<20} {'Domain':<15} {'Max Samples':>12}")
print("-" * 50)
total_samples = 0
for ds in DATASETS_TO_LOAD:
    cfg = DATASET_CONFIGS.get(ds)
    if cfg:
        print(f"{ds:<20} {cfg.domain:<15} {cfg.max_samples:>12,}")
        total_samples += cfg.max_samples
print("-" * 50)
print(f"{'TOTAL':<20} {'':<15} {total_samples:>12,}")
print(f"\nEstimated tokens: ~{total_samples * 1500 // 1_000_000_000:.1f}B")

In [ ]:
# Cell 7: Load All Datasets

print("Loading multi-domain datasets...")
print("This may take a few minutes on first run (downloading from HuggingFace)...\n")

# Load training datasets
train_dataset, train_stats = load_all_datasets(
    dataset_names=DATASETS_TO_LOAD,
    max_len=MAX_SEQ_LEN,
)

# Create a small validation set from training data
# (In production, use proper validation splits)
val_size = min(2000, len(train_dataset) // 10)
val_indices = random.sample(range(len(train_dataset)), val_size)

print(f"\n✓ Loaded {len(train_dataset):,} training samples")
print(f"✓ Created {val_size:,} validation samples")

# Show domain distribution
print("\nDomain weights for sampling:")
for domain, weight in train_dataset.weights.items():
    print(f"  {domain:15s}: {weight:.2%}")

In [ ]:
# Cell 8: Create DataLoaders

# Custom collate function for multi-domain batches
# CRITICAL: Clamp values here to prevent CUDA asserts
def collate_fn(batch):
    inputs = torch.stack([item['input'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    domains = [item['domain'] for item in batch]
    
    # Clamp inputs to valid vocab range [0, VOCAB_SIZE-1]
    inputs = inputs.clamp(0, VOCAB_SIZE - 1)
    
    # Mask invalid targets (values >= VOCAB_SIZE) as -100 (ignored in loss)
    invalid_mask = (targets >= VOCAB_SIZE) | (targets < 0)
    targets = targets.clone()
    targets[invalid_mask] = -100
    
    return {'input': inputs, 'target': targets, 'domains': domains}

# Create weighted sampler for balanced domain sampling
sampler = train_dataset.get_weighted_sampler()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=DATALOADER_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS and DATALOADER_WORKERS > 0,
    prefetch_factor=DATALOADER_PREFETCH if DATALOADER_WORKERS > 0 else None,
    collate_fn=collate_fn,
)

# Simple validation loader (no weighted sampling)
val_subset = torch.utils.data.Subset(train_dataset, val_indices)
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=DATALOADER_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

print(f"✓ Train batches per epoch: {len(train_loader):,}")
print(f"✓ Val batches: {len(val_loader):,}")

In [ ]:
# Cell 9: Preview Training Samples

print("Sample training data from each domain:\n")

# Get one sample from each domain
seen_domains = set()
for i in range(min(100, len(train_dataset))):
    sample = train_dataset[i]
    domain = sample['domain']
    if domain not in seen_domains:
        seen_domains.add(domain)
        # Decode first 150 bytes
        text = decode_special(sample['input'][:150].tolist())
        # Clean up for display
        text = text.replace('\n', '\\n')[:120]
        print(f"[{domain}]")
        print(f"  {text}...")
        print()
    
    if len(seen_domains) >= 6:  # Show up to 6 domains
        break

print(f"\nTotal domains in dataset: {len(train_dataset.domain_datasets)}")

In [ ]:
# Cell 10: Training Configuration

# ==== TRAINING HYPERPARAMETERS ====
# Adjust based on your scale

if MODEL_SCALE == '3B':
    TOTAL_STEPS = 100000
    LEARNING_RATE = 1e-4
    WARMUP_STEPS = 3000
elif MODEL_SCALE == '1B':
    TOTAL_STEPS = 80000
    LEARNING_RATE = 1.5e-4
    WARMUP_STEPS = 2000
elif MODEL_SCALE == '500M':
    TOTAL_STEPS = 50000
    LEARNING_RATE = 2e-4
    WARMUP_STEPS = 1500
else:  # 141M
    TOTAL_STEPS = 30000
    LEARNING_RATE = 3e-4
    WARMUP_STEPS = 1000

WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
VAL_EVERY = 1000
SAVE_EVERY = 5000

# ==== CWC ORDER TRAINING ====
# Auxiliary loss to train CWC to discriminate sequence order
CWC_ORDER_LOSS_WEIGHT = 0.1  # Weight for order classification loss
CWC_ORDER_EVERY = 4          # Compute order loss every N steps

# Domain loss weights (prioritize reasoning)
DOMAIN_LOSS_WEIGHTS = {
    'reasoning': 1.5,
    'assistant': 1.2,
    'code': 1.0,
    'tool_use': 1.3,
    'multilingual': 1.0,
    'docs': 0.8,
    'data': 0.8,
}

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

# Learning rate scheduler with warmup
def get_lr(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    else:
        progress = (step - WARMUP_STEPS) / (TOTAL_STEPS - WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

# Mixed precision
scaler = GradScaler(enabled=(device == 'cuda'))

print(f"Training configuration for {MODEL_SCALE}:")
print(f"  Total steps: {TOTAL_STEPS:,}")
print(f"  Warmup steps: {WARMUP_STEPS:,}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Gradient clip: {GRAD_CLIP}")
print(f"  CWC order loss weight: {CWC_ORDER_LOSS_WEIGHT}")

In [ ]:
# Cell 11: Validation Function + CWC Order Loss + Generation Demo

@torch.no_grad()
def validate(model, val_loader, device, max_batches=50):
    """Compute validation metrics across domains."""
    model.eval()
    
    total_loss = 0
    total_correct = 0
    total_tokens = 0
    domain_metrics = {}
    
    for i, batch in enumerate(val_loader):
        if i >= max_batches:
            break
        
        # Data is already clamped in collate_fn
        input_bytes = batch['input'].to(device)
        target_bytes = batch['target'].to(device)
        domains = batch['domains']
        
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            output = model(input_bytes, target_bytes)
        
        loss = output['loss'].item()
        total_loss += loss
        
        # Compute accuracy
        preds = output['logits'].argmax(dim=-1)
        mask = target_bytes != -100
        correct = (preds == target_bytes) & mask
        total_correct += correct.sum().item()
        total_tokens += mask.sum().item()
        
        # Track per-domain metrics
        for j, domain in enumerate(domains):
            if domain not in domain_metrics:
                domain_metrics[domain] = {'loss': 0, 'count': 0}
            domain_metrics[domain]['loss'] += loss / len(domains)
            domain_metrics[domain]['count'] += 1
    
    avg_loss = total_loss / max(1, min(len(val_loader), max_batches))
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    perplexity = math.exp(min(avg_loss, 10))
    
    model.train()
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'perplexity': perplexity,
        'domain_metrics': {k: v['loss']/v['count'] for k, v in domain_metrics.items() if v['count'] > 0},
    }


def compute_cwc_order_loss(model, input_bytes: Tensor, device) -> Tensor:
    """
    Compute CWC order discrimination loss.
    
    Creates shuffled sequences and trains CWC to distinguish
    correct order (label=1) from shuffled (label=0).
    """
    batch_size, seq_len = input_bytes.shape
    
    # Create shuffled version
    shuffled_bytes = input_bytes.clone()
    for i in range(batch_size):
        perm = torch.randperm(seq_len, device=device)
        shuffled_bytes[i] = input_bytes[i, perm]
    
    # Get order scores for both
    with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
        # Encode original (correct order)
        is_special = input_bytes >= 256
        cse_input = input_bytes.clone()
        cse_input[is_special] = 0
        semantic_wave = model.cse.encode_bytes(cse_input)
        wave = semantic_wave.full
        if is_special.any():
            special_indices = (input_bytes - 256).clamp(0, model.num_special_tokens - 1)
            special_embeds = model.special_token_embed(special_indices)
            wave = torch.where(is_special.unsqueeze(-1).expand_as(wave), special_embeds, wave)
        
        causal_wave_orig, order_score_orig = model.cwc(wave, return_order_score=True)
        
        # Encode shuffled (wrong order)
        is_special_shuf = shuffled_bytes >= 256
        cse_input_shuf = shuffled_bytes.clone()
        cse_input_shuf[is_special_shuf] = 0
        semantic_wave_shuf = model.cse.encode_bytes(cse_input_shuf)
        wave_shuf = semantic_wave_shuf.full
        if is_special_shuf.any():
            special_indices_shuf = (shuffled_bytes - 256).clamp(0, model.num_special_tokens - 1)
            special_embeds_shuf = model.special_token_embed(special_indices_shuf)
            wave_shuf = torch.where(is_special_shuf.unsqueeze(-1).expand_as(wave_shuf), special_embeds_shuf, wave_shuf)
        
        causal_wave_shuf, order_score_shuf = model.cwc(wave_shuf, return_order_score=True)
    
    # Binary classification: orig=1, shuffled=0
    scores = torch.cat([order_score_orig, order_score_shuf], dim=0)
    labels = torch.cat([
        torch.ones(batch_size, 1, device=device),
        torch.zeros(batch_size, 1, device=device),
    ], dim=0)
    
    order_loss = F.binary_cross_entropy_with_logits(scores, labels)
    return order_loss


@torch.no_grad()
def validation_generation_demo(model, device, step: int):
    """
    Generate sample outputs for checkpoint validation.
    Shows model's current generation quality across domains.
    """
    model.eval()
    
    demo_prompts = [
        ("assistant", "<|user|>What is 2+2?<|end|>\n<|assistant|>"),
        ("reasoning", "<|problem|>\nIf x = 5, what is 2x + 3?\n<|end|>\n<|reasoning|>\n"),
        ("code", "<|lang:python|><|code|>\ndef hello():\n    \"\"\"Print hello world.\"\"\"\n"),
    ]
    
    print(f"\n{'='*60}")
    print(f"[Step {step:,}] Validation Generation Demo")
    print(f"{'='*60}")
    
    for domain, prompt in demo_prompts:
        try:
            output = model.generate(prompt, GenerationConfigUniversal(
                max_new_bytes=80,
                temperature=0.7 if domain != 'code' else 0.3,
                domain=domain,
            ))
            generated = output[len(prompt):120]
            # Clean up for display
            generated = generated.replace('\n', '\\n')[:80]
            print(f"\n[{domain.upper()}] {prompt[:40]}...")
            print(f"  → {generated}...")
        except Exception as e:
            print(f"\n[{domain.upper()}] Generation error: {e}")
    
    print(f"{'='*60}\n")
    model.train()
    return True


# Test validation
print("Testing validation function...")
val_metrics = validate(model, val_loader, device, max_batches=5)
print(f"  Initial val loss: {val_metrics['loss']:.4f}")
print(f"  Initial perplexity: {val_metrics['perplexity']:.2f}")

# Test CWC order loss
print("\nTesting CWC order loss computation...")
test_batch = next(iter(train_loader))
test_input = test_batch['input'].to(device)
order_loss = compute_cwc_order_loss(model, test_input, device)
print(f"  Initial order loss: {order_loss.item():.4f}")

In [ ]:
# Cell 12: Checkpoint Save/Resume Functions + Disk Space Management

import shutil

CKPT_DIR = ROOT / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# ==== DISK SPACE MANAGEMENT ====
def get_disk_usage():
    """Get disk usage in GB."""
    import subprocess
    result = subprocess.run(['df', '-BG', '/'], capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    if len(lines) >= 2:
        parts = lines[1].split()
        used = float(parts[2].replace('G', ''))
        avail = float(parts[3].replace('G', ''))
        return used, avail
    return 0, 0

def cleanup_old_checkpoints(ckpt_dir: Path, keep_last: int = 1, prefix: str = 'flux_lm_universal_resume'):
    """Aggressively clean up old checkpoints to save disk space."""
    checkpoints = sorted(
        ckpt_dir.glob(f'{prefix}*.pt'),
        key=lambda p: p.stat().st_mtime
    )
    
    removed = 0
    for ckpt in checkpoints[:-keep_last] if keep_last > 0 else checkpoints:
        try:
            size_mb = ckpt.stat().st_size / 1e6
            ckpt.unlink()
            removed += 1
            print(f"  🗑 Deleted: {ckpt.name} ({size_mb:.1f} MB)")
        except Exception as e:
            print(f"  ⚠ Failed to delete {ckpt.name}: {e}")
    
    return removed

def cleanup_cache():
    """Clean up Python and PyTorch caches."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def check_disk_space_warning(min_free_gb: float = 10.0):
    """Warn if disk space is getting low."""
    try:
        _, avail = get_disk_usage()
        if avail < min_free_gb:
            print(f"  ⚠ LOW DISK SPACE: {avail:.1f} GB free (< {min_free_gb} GB)")
            return True
    except:
        pass
    return False


def save_training_checkpoint(
    path: str,
    model,
    optimizer,
    scheduler,
    scaler,
    global_step: int,
    epoch: int,
    best_val_loss: float,
    train_losses: list,
    val_losses: list,
    copy_to_drive: bool = True,
):
    """Save complete training state for resumption."""
    # Check disk space before saving
    check_disk_space_warning()
    
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'global_step': global_step,
        'epoch': epoch,
        'best_val_loss': best_val_loss,
        'train_losses': train_losses[-5000:],  # Keep last 5K only (reduced from 10K)
        'val_losses': val_losses,
        'model_scale': MODEL_SCALE,
        'config': model.config,
        'rng_state': {
            'python': random.getstate(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
        'timestamp': datetime.now().isoformat(),
    }
    torch.save(checkpoint, path)
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  ✓ Checkpoint saved: {Path(path).name} ({size_mb:.1f} MB)")
    
    # ==== COPY TO GOOGLE DRIVE ====
    if copy_to_drive and SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        try:
            drive_path = DRIVE_CKPT_DIR / Path(path).name
            shutil.copy2(path, drive_path)
            print(f"  ✓ Copied to Google Drive: {drive_path.name}")
        except Exception as e:
            print(f"  ⚠ Failed to copy to Drive: {e}")
    
    # ==== IMMEDIATE CLEANUP: Delete old local checkpoints ====
    cleanup_old_checkpoints(CKPT_DIR, keep_last=1)
    cleanup_cache()


def save_model_to_drive(model, filename: str):
    """Save model checkpoint to both local and Google Drive."""
    local_path = CKPT_DIR / filename
    
    # Delete existing file first to free space
    if local_path.exists():
        local_path.unlink()
    
    model.save(str(local_path))
    size_mb = local_path.stat().st_size / 1e6
    print(f"  ✓ Model saved: {filename} ({size_mb:.1f} MB)")
    
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        try:
            drive_path = DRIVE_CKPT_DIR / filename
            shutil.copy2(local_path, drive_path)
            print(f"  ✓ Copied to Google Drive: {filename}")
        except Exception as e:
            print(f"  ⚠ Failed to copy to Drive: {e}")


def load_training_checkpoint(path: str, model, optimizer, scheduler, scaler, device):
    """Load complete training state for resumption."""
    print(f"Loading checkpoint from {path}...")
    checkpoint = torch.load(path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    if scaler and checkpoint.get('scaler_state_dict'):
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    rng = checkpoint.get('rng_state', {})
    if rng.get('python'):
        random.setstate(rng['python'])
    if rng.get('torch') is not None:
        torch.set_rng_state(rng['torch'])
    if rng.get('cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng['cuda'])
    
    print(f"  ✓ Resumed from step {checkpoint['global_step']:,}")
    print(f"  ✓ Best val loss so far: {checkpoint['best_val_loss']:.4f}")
    
    return (
        checkpoint['global_step'],
        checkpoint['epoch'],
        checkpoint['best_val_loss'],
        checkpoint.get('train_losses', []),
        checkpoint.get('val_losses', []),
    )


def find_latest_checkpoint(ckpt_dir: Path, prefix: str = 'flux_lm_universal_resume') -> Path:
    """Find the latest resume checkpoint (checks Drive first if available)."""
    # Check Google Drive first if available
    if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
        drive_ckpts = list(DRIVE_CKPT_DIR.glob(f'{prefix}*.pt'))
        if drive_ckpts:
            def get_step(p):
                try:
                    return int(p.stem.split('_step')[-1])
                except:
                    return 0
            drive_ckpts.sort(key=get_step, reverse=True)
            print(f"  ✓ Found checkpoint on Google Drive: {drive_ckpts[0].name}")
            return drive_ckpts[0]
    
    # Check local directory
    checkpoints = list(ckpt_dir.glob(f'{prefix}*.pt'))
    if not checkpoints:
        return None
    def get_step(p):
        try:
            return int(p.stem.split('_step')[-1])
        except:
            return 0
    checkpoints.sort(key=get_step, reverse=True)
    return checkpoints[0]


# ==== INITIAL CLEANUP ====
print("Performing initial disk cleanup...")
cleanup_old_checkpoints(CKPT_DIR, keep_last=0, prefix='flux_lm_universal_resume')  # Delete ALL old resume checkpoints
cleanup_cache()

# Check disk space
try:
    used, avail = get_disk_usage()
    print(f"  Disk: {used:.1f} GB used, {avail:.1f} GB available")
except:
    print("  Disk: (unable to check)")

# Check for existing checkpoint
RESUME_TRAINING = False  # Set to True to resume from checkpoint
RESUME_CHECKPOINT = None

# Check both local and Drive for checkpoints
latest_ckpt = find_latest_checkpoint(CKPT_DIR)
if latest_ckpt and RESUME_TRAINING:
    RESUME_CHECKPOINT = str(latest_ckpt)
    print(f"✓ Found checkpoint to resume: {latest_ckpt.name}")
elif latest_ckpt:
    print(f"  ⚠ Found checkpoint {latest_ckpt.name} - set RESUME_TRAINING=True to resume")
else:
    print("  No existing checkpoint found, will train from scratch")

# Show Google Drive status
if IN_COLAB and SAVE_TO_DRIVE:
    print(f"\n📁 Google Drive checkpoint directory: {DRIVE_CKPT_DIR}")
    existing_drive_ckpts = list(DRIVE_CKPT_DIR.glob('*.pt')) if DRIVE_CKPT_DIR.exists() else []
    if existing_drive_ckpts:
        print(f"   Existing checkpoints on Drive: {len(existing_drive_ckpts)}")
        for ckpt in sorted(existing_drive_ckpts)[-3:]:  # Show last 3
            print(f"   - {ckpt.name}")

## Training Loop

Main training loop with:
1. **Multi-domain data loading** with weighted sampling
2. **Domain-weighted loss** (reasoning gets 1.5x weight)
3. **CWC order training** — auxiliary loss to teach correct sequence order
4. **Validation generation demos** at each checkpoint showing model capability
5. **Gradient accumulation** for large effective batch sizes
6. **Mixed precision** (FP16/BF16) for speed
7. **Resume capability** with full state checkpointing
8. **Google Drive saving** — all checkpoints + step models saved to Drive
9. **Aggressive disk management:**
   - Only keeps 1 local checkpoint (Drive has all)
   - GPU cache cleanup every 500 steps
   - Disk space monitor every 2500 steps
   - Emergency cleanup if <5GB free

In [ ]:
# Cell 13: Training Loop (with CWC order training + validation generation demos)

print("=" * 60)
print(f"Starting FLUX-LM Universal Training ({MODEL_SCALE})")
print("=" * 60)

if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    print(f"📁 Checkpoints will be saved to Google Drive: {DRIVE_CKPT_DIR}")

# Training state initialization
model.train()
global_step = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
order_losses = []  # Track CWC order losses
start_epoch = 0

# Resume from checkpoint if available
if RESUME_CHECKPOINT:
    global_step, start_epoch, best_val_loss, train_losses, val_losses = load_training_checkpoint(
        RESUME_CHECKPOINT, model, optimizer, scheduler, scaler, device
    )
    print(f"  Resuming training from step {global_step:,}")
else:
    print("  Starting fresh training run")

# Progress bar
pbar = tqdm(initial=global_step, total=TOTAL_STEPS, desc="Training")

# Training loop
start_time = time.time()
epoch = start_epoch
domain_losses = {}

# ==== MEMORY MANAGEMENT SETTINGS ====
CLEANUP_EVERY = 500          # Clean GPU cache every N steps
DISK_CHECK_EVERY = 2500      # Check disk space every N steps

while global_step < TOTAL_STEPS:
    epoch += 1
    
    for batch in train_loader:
        if global_step >= TOTAL_STEPS:
            break
        
        # Data is already clamped in collate_fn
        input_bytes = batch['input'].to(device, non_blocking=True)
        target_bytes = batch['target'].to(device, non_blocking=True)
        domains = batch['domains']
        
        # Forward pass with mixed precision
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            output = model(input_bytes, target_bytes)
            base_loss = output['loss']
            
            # Apply domain-specific loss weight
            domain_weight = sum(
                DOMAIN_LOSS_WEIGHTS.get(d, 1.0) for d in domains
            ) / len(domains)
            
            total_loss = (base_loss * domain_weight) / GRAD_ACCUM
            
            # ==== CWC ORDER LOSS (auxiliary) ====
            if global_step % CWC_ORDER_EVERY == 0:
                order_loss = compute_cwc_order_loss(model, input_bytes, device)
                total_loss = total_loss + (CWC_ORDER_LOSS_WEIGHT * order_loss) / GRAD_ACCUM
                order_losses.append(order_loss.item())
                # Keep bounded
                if len(order_losses) > 5000:
                    order_losses = order_losses[-2500:]
        
        # Backward pass
        scaler.scale(total_loss).backward()
        
        # Gradient accumulation
        if (global_step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        
        # Logging
        global_step += 1
        loss_val = base_loss.item()
        train_losses.append(loss_val)
        
        # ==== PERIODIC GPU CACHE CLEANUP ====
        if global_step % CLEANUP_EVERY == 0:
            cleanup_cache()
        
        # ==== PERIODIC DISK SPACE CHECK ====
        if global_step % DISK_CHECK_EVERY == 0:
            if check_disk_space_warning(min_free_gb=5.0):
                # Emergency cleanup
                cleanup_old_checkpoints(CKPT_DIR, keep_last=0)
                cleanup_cache()
        
        # Track per-domain losses (limit memory)
        for d in set(domains):
            if d not in domain_losses:
                domain_losses[d] = []
            domain_losses[d].append(loss_val)
            # Keep only last 1000 per domain to save memory
            if len(domain_losses[d]) > 1000:
                domain_losses[d] = domain_losses[d][-1000:]
        
        # Keep train_losses bounded
        if len(train_losses) > 10000:
            train_losses = train_losses[-5000:]
        
        # Update progress bar
        pbar.update(1)
        order_str = f" ord:{order_losses[-1]:.3f}" if order_losses else ""
        pbar.set_postfix({
            'loss': f"{loss_val:.4f}{order_str}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}",
            'domain': domains[0][:8],
        })
        
        # Validation
        if global_step % VAL_EVERY == 0 or global_step == TOTAL_STEPS:
            val_metrics = validate(model, val_loader, device)
            val_losses.append(val_metrics['loss'])
            
            elapsed = time.time() - start_time
            hours = elapsed / 3600
            
            # Get memory usage
            if torch.cuda.is_available():
                mem_used = torch.cuda.memory_allocated() / 1e9
                mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
                mem_str = f" | VRAM: {mem_used:.1f}/{mem_total:.1f}GB"
            else:
                mem_str = ""
            
            # CWC order loss avg
            avg_order = sum(order_losses[-100:]) / max(1, len(order_losses[-100:])) if order_losses else 0
            
            print(f"\n[Step {global_step:,}] "
                  f"Val Loss: {val_metrics['loss']:.4f} | "
                  f"Acc: {val_metrics['accuracy']:.2%} | "
                  f"PPL: {val_metrics['perplexity']:.2f} | "
                  f"Order: {avg_order:.4f} | "
                  f"Time: {hours:.2f}h{mem_str}")
            
            # Show per-domain validation losses
            if val_metrics.get('domain_metrics'):
                domain_str = ' | '.join([f"{k[:4]}:{v:.3f}" for k, v in 
                                         sorted(val_metrics['domain_metrics'].items())[:5]])
                print(f"  Domains: {domain_str}")
            
            # Save best model (with Google Drive copy)
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                save_model_to_drive(model, f'Flux-LM-Universal-{MODEL_SCALE}-best.pt')
                print(f"  ✓ New best model saved!")
        
        # ==== PERIODIC CHECKPOINT + GENERATION DEMO ====
        if global_step % SAVE_EVERY == 0:
            # Save resume checkpoint (with Google Drive copy)
            resume_path = str(CKPT_DIR / f'flux_lm_universal_resume_step{global_step}.pt')
            save_training_checkpoint(
                resume_path, model, optimizer, scheduler, scaler,
                global_step, epoch, best_val_loss, train_losses, val_losses,
                copy_to_drive=True,
            )
            
            # ==== VALIDATION GENERATION DEMO ====
            # Show model's generation capability at this checkpoint
            validation_generation_demo(model, device, global_step)
            
            # Also save current model to Drive (not just resume state)
            save_model_to_drive(model, f'Flux-LM-Universal-{MODEL_SCALE}-step{global_step}.pt')

pbar.close()

# ==== FINAL SAVES ====
print("\n" + "=" * 60)
print("Saving final checkpoints...")
print("=" * 60)

# Clean up before final save
cleanup_cache()
cleanup_old_checkpoints(CKPT_DIR, keep_last=0, prefix='flux_lm_universal_resume')

# Save final model (with Google Drive copy)
save_model_to_drive(model, f'Flux-LM-Universal-{MODEL_SCALE}.pt')

# Save final resume checkpoint (with Google Drive copy)
final_resume_path = str(CKPT_DIR / f'flux_lm_universal_resume_step{global_step}.pt')
save_training_checkpoint(
    final_resume_path, model, optimizer, scheduler, scaler,
    global_step, epoch, best_val_loss, train_losses, val_losses,
    copy_to_drive=True,
)

# Final generation demo
validation_generation_demo(model, device, global_step)

total_time = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"Training complete!")
print(f"  Model: FLUX-LM Universal {MODEL_SCALE}")
print(f"  Total time: {total_time / 3600:.2f} hours")
print(f"  Final val loss: {val_losses[-1]:.4f}")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Final order loss: {order_losses[-1]:.4f}" if order_losses else "")

# Final disk space report
try:
    used, avail = get_disk_usage()
    print(f"  Disk: {avail:.1f} GB available")
except:
    pass

if SAVE_TO_DRIVE and DRIVE_CKPT_DIR:
    print(f"\n📁 Checkpoints saved to Google Drive:")
    print(f"   {DRIVE_CKPT_DIR}")
    drive_files = list(DRIVE_CKPT_DIR.glob('*.pt'))
    for f in sorted(drive_files)[-5:]:
        size_mb = f.stat().st_size / 1e6
        print(f"   - {f.name} ({size_mb:.1f} MB)")

print(f"{'=' * 60}")

In [ ]:
# Cell 14: Plot Training Curves

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss (smoothed)
ax1 = axes[0]
window = 100
smoothed = [sum(train_losses[max(0,i-window):i+1])/min(i+1,window) 
            for i in range(len(train_losses))]
ax1.plot(smoothed, alpha=0.8)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title(f'Training Loss - FLUX-LM Universal {MODEL_SCALE}')
ax1.grid(True, alpha=0.3)

# Validation loss
ax2 = axes[1]
val_steps = list(range(VAL_EVERY, len(val_losses) * VAL_EVERY + 1, VAL_EVERY))
ax2.plot(val_steps, val_losses, 'o-', markersize=4)
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')
ax2.set_title('Validation Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
output_dir = ROOT / 'output'
output_dir.mkdir(exist_ok=True)
plt.savefig(str(output_dir / f'flux_lm_universal_{MODEL_SCALE}_curves.png'), dpi=150)
plt.show()

print(f"✓ Training curves saved to output/flux_lm_universal_{MODEL_SCALE}_curves.png")

In [ ]:
# Cell 15: Load Best Model for Evaluation

print(f"Loading best model...")
model = FluxLMUniversal.load(str(CKPT_DIR / f'Flux-LM-Universal-{MODEL_SCALE}-best.pt'), device=device)
model.eval()

# Final validation
final_metrics = validate(model, val_loader, device, max_batches=100)
print(f"\nFinal Evaluation:")
print(f"  Loss: {final_metrics['loss']:.4f}")
print(f"  Accuracy: {final_metrics['accuracy']:.2%}")
print(f"  Perplexity: {final_metrics['perplexity']:.2f}")

## Multi-Domain Generation Tests

Testing generation across different domains:
- Assistant (conversation)
- Reasoning (step-by-step)
- Code (Python)
- Multilingual

In [ ]:
# Cell 16: Multi-Domain Generation Tests

model.eval()

# ==== ASSISTANT DOMAIN ====
print("=" * 70)
print("ASSISTANT DOMAIN")
print("=" * 70)

assistant_prompts = [
    "<|system|>You are a helpful assistant.<|end|>\n<|user|>What is machine learning?<|end|>\n<|assistant|>",
    "<|user|>Explain quantum computing in simple terms.<|end|>\n<|assistant|>",
]

for prompt in assistant_prompts:
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=150,
        temperature=0.7,
        top_p=0.9,
        domain='assistant',
    ))
    print(f"\nPrompt: {prompt[:60]}...")
    print(f"Output: {output[len(prompt):200]}...")

In [ ]:
# Cell 17: Reasoning Domain Tests

print("\n" + "=" * 70)
print("REASONING DOMAIN")
print("=" * 70)

reasoning_prompts = [
    "<|problem|>\nSarah has 15 apples. She gives 4 to Tom and buys 7 more. How many apples does she have?\n<|end|>\n<|reasoning|>\n",
    "<|problem|>\nIf all roses are flowers, and some flowers fade quickly, can we conclude that some roses fade quickly?\n<|end|>\n<|reasoning|>\n",
]

for prompt in reasoning_prompts:
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=200,
        temperature=0.5,
        top_p=0.9,
        domain='reasoning',
    ))
    print(f"\nProblem: {prompt[15:80]}...")
    print(f"Reasoning: {output[len(prompt):250]}...")

In [ ]:
# Cell 18: Code Domain Tests

print("\n" + "=" * 70)
print("CODE DOMAIN")
print("=" * 70)

code_prompts = [
    "<|lang:python|><|code|>\ndef fibonacci(n):\n    \"\"\"Return the nth Fibonacci number.\"\"\"\n",
    "<|lang:python|><|code|>\ndef quicksort(arr):\n    \"\"\"Sort array using quicksort algorithm.\"\"\"\n",
]

for prompt in code_prompts:
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=200,
        temperature=0.3,  # Lower temp for code
        top_k=10,
        domain='code',
    ))
    print(f"\nPrompt: {prompt[:50]}...")
    print(f"Generated:\n{output[len(prompt):300]}")

In [ ]:
# Cell 19: Multilingual Tests

print("\n" + "=" * 70)
print("MULTILINGUAL DOMAIN")
print("=" * 70)

multilingual_prompts = [
    ("English→French", "<|user|>Translate from en to fr:\nHello, how are you today?<|end|>\n<|assistant|>"),
    ("English→German", "<|user|>Translate from en to de:\nThe weather is beautiful today.<|end|>\n<|assistant|>"),
    ("English→Spanish", "<|user|>Translate from en to es:\nI love learning new languages.<|end|>\n<|assistant|>"),
]

for name, prompt in multilingual_prompts:
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=100,
        temperature=0.5,
        domain='multilingual',
    ))
    print(f"\n{name}:")
    print(f"  Input: {prompt[40:80]}...")
    print(f"  Output: {output[len(prompt):100]}")

In [ ]:
# Cell 20: Save Final Model as .flx

print("\n" + "=" * 60)
print("Saving Final Model")
print("=" * 60)

# Save as .pt
pt_path = CKPT_DIR / f'Flux-LM-Universal-{MODEL_SCALE}.pt'
model.save(str(pt_path))
print(f"✓ Saved: {pt_path}")

# Save as .flx (FLUX ecosystem format)
flx_path = CKPT_DIR / f'Flux-LM-Universal-{MODEL_SCALE}.flx'
model.save_to_flx(str(flx_path))
print(f"✓ Saved: {flx_path}")

# File sizes
pt_size = pt_path.stat().st_size / 1e6
flx_size = flx_path.stat().st_size / 1e6
print(f"\nFile sizes:")
print(f"  {pt_path.name}: {pt_size:.1f} MB")
print(f"  {flx_path.name}: {flx_size:.1f} MB")

In [ ]:
# Cell 21: Generate Results Summary

results_md = f"""# FLUX-LM Universal Training Results

## Model
- **Architecture:** FLUX-LM Universal (Multi-Domain)
- **Scale:** {MODEL_SCALE}
- **Total Parameters:** {format_params(model.count_parameters()['total'])}
- **Vocabulary:** {model.vocab_size} (256 bytes + {model.vocab_size - 256} special tokens)
- **Training Time:** {total_time / 3600:.2f} hours
- **Device:** {device}

## Parameter Breakdown
| Component | Parameters |
|-----------|------------|
| CSE | {format_params(params['cse'])} |
| CWC | {format_params(params['cwc'])} |
| WavePredictor | {format_params(params['predictor'])} |
| Decoder | {format_params(params['decoder'])} |
| Domain Embed | {format_params(params['domain_embed'])} |
| **Total** | **{format_params(params['total'])}** |

## Training Data
| Dataset | Samples | Domain |
|---------|---------|--------|
""" + '\n'.join([f"| {name} | {count:,} | {DATASET_CONFIGS[name].domain} |" 
                 for name, count in train_stats.items()]) + f"""
| **Total** | **{sum(train_stats.values()):,}** | - |

## Training Metrics
- **Total Steps:** {global_step:,}
- **Final Val Loss:** {val_losses[-1]:.4f}
- **Best Val Loss:** {best_val_loss:.4f}
- **Final Accuracy:** {final_metrics['accuracy']:.2%}
- **Final Perplexity:** {final_metrics['perplexity']:.2f}

## Domains Trained
- TEXT: Assistant, Reasoning, Tool Use
- CODE: Python (CodeSearchNet, HumanEval, MBPP)
- MULTILINGUAL: EN-FR, EN-DE, EN-ZH, EN-ES, EN-JA
- DOCS: WikiText-103

## Key Achievements
- ✅ Multi-domain vocabulary-free generation
- ✅ Extended vocabulary with special tokens
- ✅ Domain-aware generation
- ✅ Reasoning capability (Opus trained)
- ✅ Code generation (Python focused)
- ✅ Multilingual support (5 language pairs)

## Checkpoints
- `checkpoints/Flux-LM-Universal-{MODEL_SCALE}.pt`
- `checkpoints/Flux-LM-Universal-{MODEL_SCALE}.flx`

---
*Generated: {datetime.now().isoformat()}*
"""

results_path = ROOT / 'output' / f'RESULTS_FLUX_LM_UNIVERSAL_{MODEL_SCALE}.md'
results_path.parent.mkdir(exist_ok=True)
results_path.write_text(results_md)

print(results_md)
print(f"\n✓ Results saved to {results_path}")

In [ ]:
# Cell 22: Interactive Demo

print("\n" + "=" * 60)
print("Interactive Multi-Domain Generation")
print("=" * 60)

demo_prompts = [
    ("assistant", "<|user|>What is the capital of France?<|end|>\n<|assistant|>"),
    ("reasoning", "<|problem|>\nIf x + 5 = 12, what is x?\n<|end|>\n<|reasoning|>\n"),
    ("code", "<|lang:python|><|code|>\n# Function to check if a number is prime\ndef is_prime(n):\n"),
]

for domain, prompt in demo_prompts:
    print(f"\n[{domain.upper()}]")
    print(f"Prompt: {prompt[:60]}...")
    
    output = model.generate(prompt, GenerationConfigUniversal(
        max_new_bytes=150,
        temperature=0.7 if domain != 'code' else 0.3,
        domain=domain,
    ))
    
    print(f"Output: {output[len(prompt):200]}")
    print("-" * 50)

## Training Complete!

**FLUX-LM Universal** trained on ~2B tokens across multiple domains:

| Domain | Datasets | Samples | Capability |
|--------|----------|---------|------------|
| TEXT | OpenAssistant, Dolly, Alpaca | ~227K | Conversation, instructions |
| REASONING | Opus, GSM8K, ARC | ~14K | Step-by-step problem solving |
| CODE | CodeSearchNet, HumanEval, MBPP | ~501K | Python code generation |
| MULTILINGUAL | OPUS-100 (5 pairs × 100K) | ~500K | Translation EN↔FR/DE/ZH/ES/JA |
| DOCS | WikiText-103 | ~100K | General knowledge |

**Model Selection:**
- **40GB+ VRAM** → 1B model (~930M params)
- **24-40GB VRAM** → 500M model (~480M params)

**Key Files:**
- `checkpoints/Flux-LM-Universal-{scale}.pt` - PyTorch checkpoint
- `checkpoints/Flux-LM-Universal-{scale}.flx` - FLUX ecosystem format
- `output/RESULTS_FLUX_LM_UNIVERSAL_{scale}.md` - Training results

**Next Steps:**
1. Fine-tune on specific tasks (reasoning, code)
2. Benchmark against token LLMs
3. Scale to 3B with more data and H100